### Import Data

In [6]:
import pandas as pd
import os
from pathlib import Path

In [11]:
job_titles = pd.read_csv(os.path.join(os.getcwd(), "team10_data_2026-10-15\\processed\\A_job_titles\\job_titles.csv"))
knowledge_level = pd.read_csv(os.path.join(os.getcwd(), "team10_data_2026-10-15\\processed\\B_skill_knowledge_vectors\\knowledge_level.csv"))
skill_level = pd.read_csv(os.path.join(os.getcwd(), "team10_data_2026-10-15\\processed\\B_skill_knowledge_vectors\\skill_level.csv"))

In [16]:
job_titles.head()

,job_title,onet_code
0,Chief Executives,11-1011.00
1,Chief Sustainability Officers,11-1011.03
2,General and Operations Managers,11-1021.00
3,Legislators,11-1031.00
4,Advertising and Promotions Managers,11-2011.00


### Convert Job Title(s) to Skills Vector

In [18]:
# Take a list of previous jobs and convert it to a vector of skills by taking the max skill across jobs for each skill
# Could also have a decreasing effect over time if the job was older

def get_onet_codes(titles, job_titles):
    """
    Convert a list of job titles to O*NET codes.

    Parameters
    ----------
    titles : list[str]
        Job titles held by a person.
    job_titles : pandas.DataFrame
        DataFrame containing 'job_title' and 'onet_code'.

    Returns
    -------
    list[str]
        Corresponding O*NET codes.
    """
    title_to_code = dict(
        zip(job_titles["job_title"], job_titles["onet_code"])
    )

    codes = []
    missing_titles = []

    for title in titles:
        if title in title_to_code:
            codes.append(title_to_code[title])
        else:
            missing_titles.append(title)

    if missing_titles:
        raise ValueError(
            f"Job titles not found: {missing_titles}"
        )

    return codes

In [19]:
def get_skill_vector(codes: list[str], skill_level: pd.DataFrame, knowledge_level: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate skill and knowledge ratings across jobs by taking
    the maximum value for each skill and knowledge category.

    Parameters
    ----------
    codes : list[str]
        O*NET codes for the person's jobs.
    skill_level : pandas.DataFrame
        Skill ratings, with 'onet_code' as the identifier column.
    knowledge_level : pandas.DataFrame
        Knowledge ratings, with 'onet_code' as the identifier column.

    Returns
    -------
    pandas.DataFrame
        A single-row DataFrame containing the maximum rating
        for each skill and knowledge category.
    """
    skill_rows = skill_level[
        skill_level["onet_code"].isin(codes)
    ].drop(columns="onet_code")

    knowledge_rows = knowledge_level[
        knowledge_level["onet_code"].isin(codes)
    ].drop(columns="onet_code")

    if len(skill_rows) != len(set(codes)):
        raise ValueError("One or more O*NET codes are missing from skill_level.")

    if len(knowledge_rows) != len(set(codes)):
        raise ValueError("One or more O*NET codes are missing from knowledge_level.")

    skills = skill_rows.max(axis=0)
    knowledge = knowledge_rows.max(axis=0)

    # Combine both vectors into a single row.
    result = pd.concat([skills, knowledge])

    return result.to_frame().T

In [25]:
# Test example to see the direct output
test_data = get_onet_codes(["Chief Executives"], job_titles)
output = get_skill_vector(test_data, skill_level, knowledge_level)
print(test_data)
print(output)

['11-1011.00']
   Reading Comprehension  Active Listening  Writing  Speaking  Mathematics  \
0                   4.62              4.75     4.38      4.75          3.5   

   Science  Critical Thinking  Active Learning  Learning Strategies  \
0     0.75               4.75              4.5                 3.75   

   Monitoring  ...  Education and Training  English Language  \
0        5.25  ...                     4.6              3.91   

   Foreign Language  Fine Arts  History and Archeology  \
0              1.97       0.74                    1.85   

   Philosophy and Theology  Public Safety and Security  Law and Government  \
0                     2.74                        3.35                3.58   

   Telecommunications  Communications and Media  
0                2.51                      3.25  

[1 rows x 68 columns]
